[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/anomaly_detection_partial_exercise.ipynb)

# Anomaly detection in images: finding and explaining defects

A factory photographs every hazelnut that goes through the production line, and we want to reject the defective ones. Defects are **rare** and **varied**: we have hundreds of photos of good hazelnuts, but very few of cracked or holed ones, and next month a new kind of defect may show up. A supervised classifier ("good" vs "defective") is not a good fit.

**Anomaly detection** turns the problem around: we learn what *normal* looks like, using only good images, and flag anything that does not fit. In this exercise you will:

1. **detect** defective hazelnuts (is this image anomalous?);
2. **locate** the defect (which pixels are anomalous?);
3. **explain** the decision (why is this region anomalous?).

### Model: PatchCore

We use **PatchCore** ([Roth et al., 2022](https://arxiv.org/abs/2106.08265)) from the [anomalib](https://github.com/open-edge-platform/anomalib) library:

* an ImageNet-pretrained CNN (`resnet18`) turns every image into a grid of **patch feature vectors**, one per small region of the image;
* during "training", the patch vectors of all the good images are stored in a **memory bank** (a subset of them, chosen with *coreset subsampling*, to keep it small);
* at test time, the anomaly score of a patch is the **distance to its nearest neighbour** in the memory bank, and the score of the image is (roughly) the **maximum** over its patches.

There is no gradient descent: the "model" is a pretrained feature extractor plus a set of remembered normal patches. This also makes PatchCore easy to explain: every anomalous patch can be shown next to **the most similar normal patch** the model knows.

### Dataset

[MVTec AD](https://www.mvtec.com/research-teaching/datasets/mvtec-ad) is the reference benchmark for industrial anomaly detection (license CC BY-NC-SA 4.0). We use only its `hazelnut` category: 391 good images for training and 110 test images (40 good, 70 with a `crack`, `cut`, `hole` or `print` defect), each defect with a pixel-level ground-truth mask.

### Where this fits

This notebook follows the **Week 08** notebook (PatchCore on `metal_nut`). There you *used* PatchCore as a black box and focused on the **decision**: splitting the data, choosing a threshold on a validation set and evaluating on held-out data. Here we **open the box**: what the model stores, how a score is computed, how well defects are located and, above all, **why** a region is flagged. Some settings differ on purpose (256×256 images, a 5% coreset, all the training images, anomalib 2.7.1).

> Run this notebook on **Colab with a GPU** (*Runtime → Change runtime type → T4 GPU*). The download takes a few minutes: start it right away.

In [ ]:
%pip install -q anomalib==2.7.1

If Colab asks you to **restart the session** after the installation, do it and continue from the next cell. If you ran the Week 08 notebook in the same runtime (it uses anomalib 2.2.0), **restart the session** after this installation, so that the new version is the one imported.

In [ ]:
import shutil
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from huggingface_hub import snapshot_download

from anomalib.data import MVTecAD
from anomalib.models import Patchcore
from anomalib.engine import Engine
from lightning.pytorch.callbacks import TQDMProgressBar

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using", device)

torch.manual_seed(0)


def check(condition, message):
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)

## 1. Dataset

### 1.1 Download

The official MVTec download links are currently broken, so we download only the `hazelnut` images and masks from a [Hugging Face mirror](https://huggingface.co/datasets/TheoM55/mvtec_anomaly_detection) and arrange them in the original MVTec folder layout:

```
datasets/MVTecAD/hazelnut/
    train/good/           391 good images
    test/good/            40 good images
    test/<defect>/        defective images (crack, cut, hole, print)
    ground_truth/<defect>/ one mask per defective image (white = defect)
```

In [ ]:
ROOT = Path("datasets/MVTecAD")
CATEGORY = "hazelnut"
target = ROOT / CATEGORY

if not target.exists():
    raw = Path("datasets/hf_mvtec")
    snapshot_download(
        "TheoM55/mvtec_anomaly_detection",
        repo_type="dataset",
        allow_patterns=[f"images/*/{CATEGORY}/**", f"masks/*/{CATEGORY}/**"],
        local_dir=raw,
        max_workers=16,
    )
    shutil.copytree(raw / "images" / "train" / CATEGORY, target / "train")
    shutil.copytree(raw / "images" / "test" / CATEGORY, target / "test")
    shutil.copytree(raw / "masks" / "test" / CATEGORY, target / "ground_truth")
    shutil.rmtree(raw)

for folder in sorted(target.glob("*/*")):
    print(folder, len(list(folder.glob("*.png"))))

### 1.2 The datamodule

anomalib's [`MVTecAD`](https://anomalib.readthedocs.io/en/latest/markdown/guides/reference/data/datamodules/image/mvtecad.html) datamodule builds the training and test sets and their dataloaders. Each item has the `image` (a `(3, 1024, 1024)` tensor with values in `[0, 1]`), the image label `gt_label` (`True` = anomalous) and the ground-truth mask `gt_mask`. The table `dm.test_data.samples` has one row per test image; its `label` column has the defect type (`good` for the normal ones).

In [ ]:
dm = MVTecAD(root=ROOT, category=CATEGORY, train_batch_size=32, eval_batch_size=32, num_workers=2)
dm.setup()

print(len(dm.train_data), "training images,", len(dm.test_data), "test images")
samples = dm.test_data.samples
samples.assign(file=samples["image_path"].map(lambda p: Path(p).name))[["file", "label", "label_index"]].head()

### 1.3 Explore

Count the **test** images of each defect type (including `good`). Store the result in `counts`, a dictionary or a pandas `Series` from the defect name to the number of images.

In [ ]:
# TODO: counts = number of test images per defect type
counts = ...


In [ ]:
check(len(dm.train_data) == 391, "391 training images")
check(counts["good"] == 40 and sum(counts[d] for d in ["crack", "cut", "hole", "print"]) == 70,
      "40 good and 70 defective test images")

The helper below shows one test image of each type with its ground-truth mask in red.

In [ ]:
defects = ["good", "crack", "cut", "hole", "print"]
labels = dm.test_data.samples["label"].tolist()
example = {d: labels.index(d) for d in defects}  # first test image of each type


def show(ax, image, mask=None, title=""):
    ax.imshow(image.permute(1, 2, 0).cpu())
    if mask is not None:
        ax.contour(mask.cpu(), levels=[0.5], colors="red", linewidths=1)
    ax.set_title(title)
    ax.axis("off")


fig, axes = plt.subplots(1, len(defects), figsize=(16, 3.5))
for ax, d in zip(axes, defects):
    item = dm.test_data[example[d]]
    show(ax, item.image, item.gt_mask, d)
plt.show()

**Explain (briefly):** Recall from Week 08 that the detector is fitted on good images only. In two or three sentences: what would a supervised "good vs defective" classifier need that we do not have, and what would happen to it (and to PatchCore) when a new kind of defect appears?

**Your answer:**



## 2. Model: PatchCore

### 2.1 What PatchCore stores

PatchCore resizes the images to 256×256 and takes the outputs of `layer2` (stride 8, 128 channels) and `layer3` (stride 16, 256 channels) of a `resnet18`. `layer3` is upsampled to the size of `layer2` and both are concatenated, so every image becomes a **32×32 grid of 384-dimensional patch vectors**. Each vector describes a small region of the image, together with its surroundings.

**Predict first:** In Week 08 you saw the memory bank of a smaller configuration (40 images, 128×128 pixels). Do the same arithmetic here: how many patch vectors do the 391 training images produce? If the memory bank keeps 5% of them (`coreset_sampling_ratio=0.05`), how many vectors will it have?

**Your answer:**



### 2.2 Training

Create:

* `model`: a [`Patchcore`](https://anomalib.readthedocs.io/en/latest/markdown/guides/reference/models/image/patchcore.html) with `backbone="resnet18"`, `layers=["layer2", "layer3"]` and `coreset_sampling_ratio=0.05`;
* `engine`: an [`Engine`](https://anomalib.readthedocs.io/en/latest/markdown/guides/reference/engine/index.html) with `logger=False` and `callbacks=[TQDMProgressBar()]` (anomalib's engine is a PyTorch Lightning `Trainer`, and its default progress bar does not always work in notebooks);

and call `engine.fit(model=model, datamodule=dm)`. It should take one or two minutes on a GPU, most of it in the coreset selection.

In [ ]:
# TODO: model, engine, and fit the model


In [ ]:
memory_bank = model.model.memory_bank
print("memory bank:", tuple(memory_bank.shape))
check(memory_bank.shape[1] == 384, "patch vectors have 128 + 256 = 384 dimensions")
check(abs(memory_bank.shape[0] - 0.05 * 391 * 32 * 32) < 2, "the memory bank keeps 5% of the patch vectors")

**Explain:** There was one "epoch", no optimizer and no loss curve. What did the model actually learn from the training images? The `resnet18` was trained on ImageNet and never saw a hazelnut: why are its features still useful here? Why do we use the middle layers (`layer2`, `layer3`) and not the last one (`layer4`)?

**Your answer:**



## 3. Detection: is the image anomalous?

`engine.test` computes the anomaly score of every test image and pixel and compares them with the labels and the masks:

* `image_AUROC`: how well the **image** score separates good from defective images;
* `pixel_AUROC`: how well the **pixel** scores (the anomaly map) separate defect pixels from normal pixels;
* the `F1Score`s use a threshold that anomalib chooses automatically.

Unlike in Week 08, we do **not** choose a threshold here: the AUROC is computed over all possible thresholds.

In [ ]:
results = engine.test(model=model, datamodule=dm)[0]
results

In [ ]:
check(results["image_AUROC"] > 0.95, "image AUROC above 0.95")
check(results["pixel_AUROC"] > 0.95, "pixel AUROC above 0.95")

**Explain:**

1. In Week 08 you chose a threshold by maximizing the balanced accuracy on a validation set. The AUROC needs no threshold: what does it measure instead, and why is it useful to look at it *before* choosing a threshold? (Think also of a real line where 1 in 1000 hazelnuts is defective: what is the accuracy of a detector that never flags anything?)
2. The pixel AUROC is very high: does that mean the predicted defect regions match the ground truth well? Hint: what fraction of the pixels are defect pixels?

**Your answer:**



## 4. Localization: where is the defect?

### 4.1 Anomaly maps

`engine.predict` returns, for every test image, the **anomaly map** (the patch scores upsampled to 256×256 and smoothed, normalized so that the threshold is 0.5), the **predicted mask** (`anomaly_map >= 0.5`) and the image score `pred_score`. The cell below gathers them for the whole test set. You already looked at anomaly maps in Week 08; here we need them to **measure** how well the defects are located. Note that anomalib normalizes these maps so that the threshold is 0.5, unlike the raw scores of Week 08.

In [ ]:
predictions = engine.predict(model=model, dataloaders=dm.test_dataloader())

anomaly_maps = torch.cat([b.anomaly_map for b in predictions]).cpu()  # (110, 256, 256)
pred_masks = torch.cat([b.pred_mask for b in predictions]).cpu().bool()
gt_masks = torch.cat([b.gt_mask for b in predictions]).cpu().bool()
pred_scores = torch.cat([b.pred_score for b in predictions]).cpu()
image_paths = [p for b in predictions for p in b.image_path]
check([Path(p).parent.name for p in image_paths] == labels, "predictions are in the same order as the test set")


def small(image, size=256):
    return F.interpolate(image[None], size=size, mode="bilinear", antialias=True)[0]


fig, axes = plt.subplots(3, len(defects), figsize=(16, 10))
for col, d in enumerate(defects):
    i = example[d]
    image = small(dm.test_data[i].image)
    show(axes[0, col], image, gt_masks[i], f"{d}  (score {pred_scores[i]:.2f})")
    show(axes[1, col], image, title="anomaly map")
    axes[1, col].imshow(anomaly_maps[i], cmap="jet", alpha=0.5, vmin=0, vmax=1)
    show(axes[2, col], image, pred_masks[i], "predicted mask")
plt.show()

### 4.2 Measuring the overlap

The **Intersection over Union** of a predicted mask $P$ and a ground-truth mask $G$ is

$$\text{IoU}(P, G) = \frac{|P \cap G|}{|P \cup G|}.$$

Write `iou(pred, gt)` for two boolean tensors of the same shape. If both masks are empty, return `1.0`. Then compute `iou_per_defect`: a dictionary from each defect type (`crack`, `cut`, `hole`, `print`) to the **mean** IoU of the test images of that type.

In [ ]:
# TODO: def iou(pred, gt), and iou_per_defect


In [ ]:
a = torch.zeros(4, 4, dtype=torch.bool)
b = torch.zeros(4, 4, dtype=torch.bool)
a[:2] = True   # 8 pixels
b[1:3] = True  # 8 pixels, 4 in common
check(abs(iou(a, b) - 4 / 12) < 1e-6, "iou of two half-overlapping masks is 4/12")
check(iou(a, a) == 1.0 and iou(b & ~b, b & ~b) == 1.0, "identical masks (or two empty ones) give 1")
check(sorted(iou_per_defect) == ["crack", "cut", "hole", "print"], "one mean IoU per defect type")
check(all(0.1 < v < 1 for v in iou_per_defect.values()), "every defect type is at least partly located")

**Explain:** Which defect type is located best, and which worst? Look at the anomaly maps above: is the predicted mask usually too large or too small, and why could a model that works with 8×8-pixel patches (and smooths its map) have trouble with thin cracks?

**Your answer:**



## 5. Explanation: why is this region anomalous?

The anomaly map shows **where**, but not **why**. Week 08 ended with the question of whether a heatmap identifies a physical defect or just a region unlike the normal reference; this section makes that distinction concrete. PatchCore's decision rule gives us a direct, faithful explanation: a patch is anomalous because **even the most similar normal patch in the memory bank is far from it**. So we can show the user that normal patch:

> "This region is unusual: the closest thing I have seen in good hazelnuts is *this* (shown), and it is still 3× farther away than normal regions usually are."

This is an **example-based** explanation, like explaining a k-nearest-neighbours classifier with its neighbours. Since the memory bank stores *real* training patches, we can find which training image and position each one came from.

### 5.1 Patch scores by hand

`embed(images)` (given) runs the PatchCore feature extractor on raw images (values in `[0, 1]`, any size) and returns the patch vectors as a `(B, 384, 32, 32)` tensor.

Write `patch_scores(image)` that, for **one** raw image `(3, H, W)`, returns:

* `scores`: a `(32, 32)` tensor with the Euclidean distance from each patch vector to its nearest neighbour in `memory_bank` ([`torch.cdist`](https://pytorch.org/docs/stable/generated/torch.cdist.html), then `min`);
* `nearest`: a `(32, 32)` long tensor with the **index** of that nearest neighbour in `memory_bank`.

Hint: reshape the `(1, 384, 32, 32)` embedding into `(1024, 384)` so that row `r * 32 + c` is the patch at row `r`, column `c`.

In [ ]:
pc = model.model.to(device).eval()  # the PyTorch PatchCore model inside the Lightning module
memory_bank = pc.memory_bank
transform = model.pre_processor.transform  # resize to 256x256 + ImageNet normalization


@torch.no_grad()
def embed(images):
    features = pc.feature_extractor(transform(images.to(device)))
    features = {layer: pc.feature_pooler(f) for layer, f in features.items()}
    return pc.generate_embedding(features)


print(embed(dm.test_data[0].image[None]).shape)

In [ ]:
# TODO: def patch_scores(image) -> scores, nearest


In [ ]:
image = dm.test_data[example["hole"]].image
scores, nearest = patch_scores(image)
check(scores.shape == (32, 32) and nearest.shape == (32, 32), "scores and nearest are 32x32")
vectors = embed(image[None])[0].permute(1, 2, 0).reshape(-1, 384)
ref_scores, ref_nearest = pc.nearest_neighbors(vectors, n_neighbors=1)
check(torch.allclose(scores.flatten(), ref_scores.flatten(), atol=1e-3), "same scores as anomalib")
check(bool((nearest.flatten() == ref_nearest.flatten()).float().mean() > 0.99), "same nearest neighbours as anomalib")

### 5.2 How far is "far"?

A distance of, say, 5 means nothing by itself. To make the explanation readable, compare it with the distances of **normal** patches: compute the patch scores of all the **good test images** (they are not in the memory bank), put them all in a 1-D tensor `normal_scores`, and store its 99th percentile in `p99` ([`torch.quantile`](https://pytorch.org/docs/stable/generated/torch.quantile.html)).

A patch with a score of `2 * p99` is then "twice as far from the normal data as 99% of normal patches".

In [ ]:
# TODO: normal_scores and p99


In [ ]:
check(normal_scores.shape == (40 * 32 * 32,), "one score per patch of the 40 good test images")
check(0 < p99 < scores.max().item(), "the most anomalous patch of the 'hole' image is above p99")

### 5.3 Where does a memory bank vector come from?

`find_sources(indices)` (given) recomputes the patch vectors of all the training images and, for each memory bank index, finds the training image and the patch position `(row, col)` it was copied from. It returns a dictionary `index -> (train_image_index, row, col, distance)`; the distance should be (almost) zero, since the memory bank contains exact copies of training patches.

It goes through the whole training set, so call it **once** with all the indices you need.

In [ ]:
@torch.no_grad()
def find_sources(indices):
    indices = sorted(set(int(i) for i in indices))
    queries = memory_bank[indices]
    best = torch.full((len(indices),), float("inf"), device=device)
    sources = torch.zeros(len(indices), 3, dtype=torch.long, device=device)
    loader = DataLoader(dm.train_data, batch_size=32, shuffle=False,
                        collate_fn=dm.train_data.collate_fn, num_workers=2)
    start = 0
    for batch in loader:
        embedding = embed(batch.image)
        n, dim, h, w = embedding.shape
        vectors = embedding.permute(0, 2, 3, 1).reshape(-1, dim)
        distance, j = torch.cdist(queries, vectors, compute_mode="donot_use_mm_for_euclid_dist").min(dim=1)
        better = distance < best
        best[better] = distance[better]
        position = j % (h * w)
        found = torch.stack([start + j // (h * w), position // w, position % w], dim=1)
        sources[better] = found[better]
        start += n
    return {i: (*sources[k].tolist(), best[k].item()) for k, i in enumerate(indices)}


top5 = scores.flatten().topk(5).indices  # the 5 most anomalous patches of the 'hole' image
sources = find_sources(nearest.flatten()[top5])
sources

In [ ]:
check(all(s[3] < 0.01 * p99 for s in sources.values()), "memory bank vectors are (almost) exact copies of training patches")

### 5.4 Explaining a defective hazelnut

`plot_explanation(test_index, cells, sources)` (given) draws one row per patch in `cells`, a list of `(row, col, score, memory_index)`:

1. the test image with the patch marked;
2. a zoom of the test patch;
3. a zoom of the **nearest normal patch**;
4. the training image it comes from, with that patch marked.

Each patch covers 8×8 pixels of the 256×256 image, but its vector "sees" a larger region, so the zooms show 32×32 pixels (of the 256×256 image) around it.

Now, for each defect type (use `example[d]` as the test image), write the code that:

1. computes `scores, nearest = patch_scores(...)`;
2. picks the **2 most anomalous** patches and the **least anomalous** one (as a reference), as `(row, col, score, memory_index)` tuples;

Collect the patches of all the images first, call `find_sources` **once** with all their memory indices, and then call `plot_explanation` for each image. Store the patches in `explained`, a dictionary from the test index to its list of cells.

In [ ]:
CELL, ZOOM = 8, 32  # patch size and zoom size, in pixels of the 256x256 image


def box(row, col, scale):
    centre_y, centre_x = (row + 0.5) * CELL * scale, (col + 0.5) * CELL * scale
    half = ZOOM * scale / 2
    return int(centre_y - half), int(centre_x - half), int(ZOOM * scale)


def zoom(image, row, col):
    scale = image.shape[-1] / 256
    y, x, size = box(row, col, scale)
    padded = F.pad(image, [size] * 4)  # so that border patches can be zoomed too
    return padded[:, y + size:y + 2 * size, x + size:x + 2 * size]


def mark(ax, image, row, col, colour):
    show(ax, image)
    y, x, size = box(row, col, image.shape[-1] / 256)
    ax.add_patch(mpatches.Rectangle((x, y), size, size, fill=False, edgecolor=colour, linewidth=2))


def plot_explanation(test_index, cells, sources):
    test_image = dm.test_data[test_index].image
    fig, axes = plt.subplots(len(cells), 4, figsize=(13, 3.3 * len(cells)), squeeze=False)
    for (row, col, score, index), ax in zip(cells, axes):
        train_index, train_row, train_col, _ = sources[index]
        train_image = dm.train_data[train_index].image
        colour = "red" if score > p99 else "lime"
        mark(ax[0], test_image, row, col, colour)
        ax[0].set_title(f"test {test_index} ({labels[test_index]})")
        show(ax[1], zoom(test_image, row, col), title=f"patch ({row}, {col})")
        show(ax[2], zoom(train_image, train_row, train_col),
             title=f"nearest normal: {score:.2f} = {score / p99:.1f} x p99")
        mark(ax[3], train_image, train_row, train_col, "lime")
        ax[3].set_title(f"train {train_index}, patch ({train_row}, {train_col})")
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO: explained = {test_index: [(row, col, score, memory_index), ...]}, find_sources, plot_explanation


In [ ]:
check(len(explained) == 4 and all(len(c) == 3 for c in explained.values()), "3 patches for each of the 4 defect types")
check(all(c[0][2] >= c[1][2] >= c[2][2] for c in explained.values()), "most anomalous patches first, least anomalous last")
check(all(c[0][2] > p99 for c in explained.values()), "the top patch of every defective image is above p99")

**Explain:** Look at the rows of each defect type.

1. Is the most anomalous patch on the defect (compare with the ground-truth masks of section 1.3)?
2. What does the nearest normal patch look like: a similar texture, the same part of the hazelnut (tip, border, background)? Does it help to understand *what* is wrong with the test patch?
3. Compare the least anomalous patch with its neighbour: how close are they, and why?
4. Is this explanation **faithful**, i.e. does it show the real reason for the score? Compare it with a post-hoc method such as Grad-CAM applied to a classifier.

**Your answer:**



## 6. Going further (optional)

* Compare the threshold that anomalib chose automatically (used for the F1 scores and the predicted masks) with the one you would get by maximizing the balanced accuracy on a validation set, as in Week 08. Which one would you use on the factory line, and why?
* Train a [`Padim`](https://anomalib.readthedocs.io/en/latest/markdown/guides/reference/models/image/padim.html) model, which models each patch position with a Gaussian instead of a memory bank. Compare the AUROCs and the IoUs. How would you explain a PaDiM decision?
* Change the backbone to `wide_resnet50_2` (the one of the paper) or the `coreset_sampling_ratio` (0.01, 0.1). What happens to the memory bank size, the fitting time and the metrics?
* Repeat the exercise with another category, for example `bottle` (easy) or `screw` (hard): change `CATEGORY` and re-run the notebook.
* anomalib has a [`VlmAd`](https://anomalib.readthedocs.io/en/latest/markdown/guides/reference/models/image/vlm_ad.html) model that asks a vision-language model (e.g. through Ollama or the OpenAI API) to describe the anomaly in words. How does a text explanation compare with the nearest-patch explanation?

### AI usage declaration

If you used an AI tool (ChatGPT, Copilot, Gemini, Claude, ...) in this exercise, say **which one**, **for which sections**, and **what you asked it**. You may be asked to explain or change any cell of this notebook in class.

**Your answer:**

